<a href="https://colab.research.google.com/github/lnittala1-ops/BAN482/blob/main/BAN482_Gradio_cloud_hosting_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚢 Tutorial: Build a Titanic Survival App with Gradio

In this lab, we will train a Machine Learning model to predict passenger survival on the Titanic and then wrap it in a user-friendly web app using **Gradio**.

**Learning Objectives:**
1. Clean and preprocess raw data.
2. Train a classification model (Random Forest).
3. Create a prediction function that accepts human-readable inputs.
4. Build a web interface to interact with your model.

## Step 1: Install Gradio

First, we need to install the library that will create our app. Run this in your first Colab cell.

In [ ]:
!pip install -q gradio # Install the Gradio library to build the web application.

## Step 2: Import Libraries and Load Data

We will use `pandas` for data manipulation and `sklearn` for machine learning.

In [ ]:
import pandas as pd # Import pandas for data manipulation and analysis.
import numpy as np # Import numpy for numerical operations (though not directly used in this snippet, often useful).
import gradio as gr # Import gradio for building interactive web interfaces.
from sklearn.model_selection import train_test_split # Import train_test_split to divide data for training and testing.
from sklearn.ensemble import RandomForestClassifier # Import RandomForestClassifier for building the machine learning model.

# Load the Titanic dataset from a URL into a pandas DataFrame.
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
data = pd.read_csv(url) # Read the CSV file into a DataFrame named 'data'.

print("Data loaded successfully!") # Print a confirmation message.
data.head() # Display the first few rows of the loaded DataFrame to inspect the data.

## Step 3: Data Preprocessing

Machine learning models need numbers, not words. We need to:
1. Drop columns we don't need (like Name and Ticket).
2. Fill in missing values (Age).
3. Convert categorical text (Male/Female) into numbers (0/1).
4. **One-Hot Encode 'Pclass'** to treat classes as distinct categories rather than a ranked number.

In [ ]:
def preprocess_data(df): # Define a function to preprocess the raw data.

    df = df.copy() # Create a copy of the DataFrame to avoid modifying the original data.
    # 1. Select useful features for the model and include 'Survived' for the target variable.

    features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']

    df = df[features + ['Survived']].dropna()
    # Select features and target, then drop rows with any missing values for simplicity.

    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
    # 2. Encode 'Sex' column: Convert 'male' to 0 and 'female' to 1, as machine learning models require numerical input.

    pclass_dummies = pd.get_dummies(df['Pclass'], prefix='Pclass', drop_first=True)
    # 3. One-Hot Encode 'Pclass' column to treat passenger classes as distinct categories.
    # drop_first=True removes the first category (Class 1) to prevent multicollinearity, creating 'Pclass_2' and 'Pclass_3'.
    # If both 'Pclass_2' and 'Pclass_3' are 0, it implicitly means the passenger is in Class 1.

    df = pd.concat([df, pclass_dummies], axis=1)
    # Concatenate the one-hot encoded columns back to the DataFrame.

    df.drop('Pclass', axis=1, inplace=True)
    # Drop the original 'Pclass' column as it has been replaced by the encoded versions.

    return df
    # Return the preprocessed DataFrame.

clean_data = preprocess_data(data)
# Apply the preprocessing function to the original 'data' DataFrame.

# Split the preprocessed data into features (X) and the target variable (y).

X = clean_data.drop('Survived', axis=1)
# X contains all columns except 'Survived'.

y = clean_data['Survived']
# y contains only the 'Survived' column.

print("Data cleaned with One-Hot Encoding!")
# Confirm data cleaning completion.

print("Features used:", X.columns.tolist())
# Display the list of features used in the model.

X.head()
# Show the first few rows of the features DataFrame.

## Step 4: Train the Model

We will use a **Random Forest Classifier**. It's powerful and works well for this type of data.

In [ ]:
# Split the data into training and testing sets: 80% for training, 20% for testing.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # random_state ensures reproducibility.

# Initialize the RandomForestClassifier model with 100 estimators (decision trees) and a random state for reproducibility.
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train) # Train the model using the training features (X_train) and target (y_train).

# Evaluate the model's accuracy on the test set.
accuracy = model.score(X_test, y_test)
print(f"Model Accuracy: {accuracy:.2%}") # Print the calculated accuracy, formatted as a percentage.

## Step 5: Define the Prediction Function

This is the **bridge** between the user interface and the model. The App will send inputs (like "Male", "3rd Class") to this function. This function must convert them into the numbers the model expects.

*Note: Because we One-Hot Encoded 'Pclass', we need to manually translate the user's choice (1, 2, or 3) into the `Pclass_2` and `Pclass_3` features.*

In [ ]:
def predict_survival(pclass, sex, age, sibsp, parch, fare):
# Define the prediction function that the Gradio app will call.

    # 1. Encode 'Sex' input: Convert user's 'Male'/'Female' choice to numerical (0/1) for the model.
    sex_encoded = 0 if sex == "Male" else 1

    # 2. Encode 'Pclass' using One-Hot Encoding logic, matching the training data's format.
    # The model was trained with 'Pclass_2' and 'Pclass_3' columns.
    # Class 1 is represented when both Pclass_2 and Pclass_3 are 0.
    pclass_2 = 1 if pclass == 2 else 0 # Set Pclass_2 to 1 if passenger class is 2, else 0.
    pclass_3 = 1 if pclass == 3 else 0 # Set Pclass_3 to 1 if passenger class is 3, else 0.

    # 3. Create a pandas DataFrame from the user inputs, ensuring correct column names and order.
    # The column order must exactly match the feature columns (X.columns) used during model training.
    input_data = pd.DataFrame([[sex_encoded, age, sibsp, parch, fare, pclass_2, pclass_3]],
                              columns=['Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Pclass_2', 'Pclass_3'])

    # 4. Use the trained model to make a prediction on the input data.
    prediction = model.predict(input_data) # Predict the survival outcome (0 or 1).
    probability = model.predict_proba(input_data)[0][1] # Get the probability of survival (class 1).

    # 5. Return a user-friendly message based on the prediction and probability.
    if prediction[0] == 1:
    # If the model predicts survival.
        return f"Survived! (Probability: {probability:.2%})"
        # Return a message indicating survival with probability.
    else: # If the model predicts non-survival.
        return f"Did not survive. (Probability: {probability:.2%})"
        # Return a message indicating non-survival with probability.

## Step 6: Build the Gradio App

Now we connect our `predict_survival` function to visual inputs (sliders, dropdowns).

In [ ]:
# Define the Gradio interface, connecting the prediction function to interactive input components.
demo = gr.Interface(
    fn=predict_survival,  # Specify the Python function that will process the inputs and return outputs.
    inputs=[ # Define the input components for the Gradio interface.
        gr.Dropdown([1, 2, 3], label="Passenger Class (1=1st, 3=3rd)"), # Dropdown for passenger class.
        gr.Dropdown(["Male", "Female"], label="Sex"), # Dropdown for sex.
        gr.Slider(1, 100, value=30, label="Age"), # Slider for age, with a default value of 30.
        gr.Slider(0, 8, step=1, label="Siblings/Spouses Aboard"), # Slider for number of siblings/spouses.
        gr.Slider(0, 6, step=1, label="Parents/Children Aboard"), # Slider for number of parents/children.
        gr.Slider(0, 500, value=32, label="Fare Price ($)") # Slider for fare price, with a default value of 32.
    ],
    outputs="text", # Specify that the output of the function will be displayed as text.
    title="🚢 Titanic Survival Predictor", # Set the title of the Gradio application.
    description="Enter passenger details to predict if they would have survived the Titanic." # Set the description displayed under the title.
)

# Launch the Gradio application.
demo.launch(share=True) # Set share=True to generate a publicly accessible (temporary) URL for the app.

---

# Part 2: Permanent Deployment on Hugging Face Spaces

The link above is temporary; it will stop working when this notebook closes. To host this app permanently, we will deploy it to **Hugging Face Spaces**.

To do this, we need to export our code into two files: `requirements.txt` (dependencies) and `app.py` (the code).


## Step 7: Create 'requirements.txt'
This file tells Hugging Face which libraries to install.

In [ ]:
# Create a file named 'requirements.txt' to list all necessary Python libraries for deployment.
with open("requirements.txt", "w") as f: # Open the file in write mode.
    f.write("pandas\n") # Write 'pandas' to the file, followed by a newline.
    f.write("scikit-learn\n") # Write 'scikit-learn' to the file.
    f.write("gradio\n") # Write 'gradio' to the file.

print("requirements.txt created!") # Confirm that the requirements file has been created.

## Step 8: Create 'app.py'
We will combine all our logic (data loading, training, interface) into one script using the `%%writefile` magic command.

**Note:** For simplicity, we are retraining the model every time the app starts. In a professional production environment, you would usually save the trained model (e.g., as a pickle file) and load it, but this approach ensures your app works self-contained.

In [ ]:
%%writefile app.py
# Magic command to write the following cell content into a file named 'app.py'.

import pandas as pd # Import pandas for data manipulation.
import gradio as gr # Import gradio for building the web interface.
from sklearn.ensemble import RandomForestClassifier # Import RandomForestClassifier for the machine learning model.

# --- 1. Load & Preprocess Data --- # Section for data loading and preprocessing.
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv" # URL of the Titanic dataset.
data = pd.read_csv(url) # Load the dataset into a pandas DataFrame.

def preprocess_data(df): # Define the data preprocessing function.
    df = df.copy() # Create a copy of the DataFrame.
    features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare'] # Define features to use.
    df = df[features + ['Survived']].dropna() # Select features and target, drop rows with missing values.
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1}) # Encode 'Sex' column to numerical.
    # One-Hot Encode 'Pclass' column (drop_first=True to avoid multicollinearity).
    pclass_dummies = pd.get_dummies(df['Pclass'], prefix='Pclass', drop_first=True)
    df = pd.concat([df, pclass_dummies], axis=1) # Concatenate one-hot encoded columns.
    df.drop('Pclass', axis=1, inplace=True) # Drop the original 'Pclass' column.
    return df # Return the preprocessed DataFrame.

clean_data = preprocess_data(data) # Apply preprocessing to the data.
X = clean_data.drop('Survived', axis=1) # Define features (X).
y = clean_data['Survived'] # Define target variable (y).

# --- 2. Train Model --- # Section for model training.
model = RandomForestClassifier(n_estimators=100, random_state=42) # Initialize the Random Forest model.
model.fit(X, y) # Train the model on the entire preprocessed dataset.

# --- 3. Prediction Function --- # Section for the prediction logic.
def predict_survival(pclass, sex, age, sibsp, parch, fare): # Define the prediction function.
    sex_encoded = 0 if sex == "Male" else 1 # Encode 'Sex' input.
    pclass_2 = 1 if pclass == 2 else 0 # Encode 'Pclass' for Pclass_2.
    pclass_3 = 1 if pclass == 3 else 0 # Encode 'Pclass' for Pclass_3.

    # Create a DataFrame for input, ensuring column order matches training data.
    input_data = pd.DataFrame([[sex_encoded, age, sibsp, parch, fare, pclass_2, pclass_3]],
                              columns=['Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Pclass_2', 'Pclass_3'])

    prediction = model.predict(input_data) # Get the survival prediction.
    probability = model.predict_proba(input_data)[0][1] # Get the probability of survival.

    if prediction[0] == 1: # Return message based on prediction.
        return f"Survived! (Probability: {probability:.2%})"
    else:
        return f"Did not survive. (Probability: {probability:.2%})"

# --- 4. Launch App --- # Section for launching the Gradio app.
demo = gr.Interface(
    fn=predict_survival, # Link to the prediction function.
    inputs=[ # Define input components for the app.
        gr.Dropdown([1, 2, 3], label="Passenger Class (1=1st, 3=3rd)"),
        gr.Dropdown(["Male", "Female"], label="Sex"),
        gr.Slider(1, 100, value=30, label="Age"),
        gr.Slider(0, 8, step=1, label="Siblings/Spouses Aboard"),
        gr.Slider(0, 6, step=1, label="Parents/Children Aboard"),
        gr.Slider(0, 500, value=32, label="Fare Price ($)")
    ],
    outputs="text", # Output type is text.
    title="🚢 Titanic Survival Predictor", # App title.
    description="Enter passenger details to predict if they would have survived the Titanic." # App description.
)

demo.launch() # Launch the Gradio app (without public sharing for the app.py file).

## Step 9: Upload to Hugging Face

1.  **Download Files:** Look at the folder icon on the left sidebar of Colab. You should see `app.py` and `requirements.txt`. Right-click and download them.
2.  **Create Space:** Go to [huggingface.co/spaces](https://huggingface.co/spaces) and click "Create new Space".
3.  **Setup:**
    * **Space Name:** (e.g., `titanic-predictor`)
    * **SDK:** Select **Gradio**.
4.  **Upload:** Once the Space is created, go to the **Files** tab and click **Add file > Upload files**. Drag and drop your `app.py` and `requirements.txt` there.
5.  **Commit:** Click "Commit changes to main".

Your app will now build (this takes about 1-2 minutes) and then be live on the web permanently!